# Analysis Pipeline

This notebook runs the full analysis pipeline, from feature extraction through to the statistical characterisation of APAs. It assumes the preprocessing pipeline (`preprocessing/run_preprocessing.ipynb`) has already been completed.

**Pipeline overview:**

```
Per-condition .pkl files (from preprocessing)
  |
  v
1. BasicMeasures          Extract stride-level and run-level kinematic measures
  |
  v
2. Main                   PCA, regression, clustering, and cross-validation
  |
  v
3. Learning               Temporal learning curves and APA development
  |
  v
4. WhenAPA                Stride-wise APA timing analysis
```

In [ ]:
import os
import sys
import pickle

# Ensure project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.abspath(""))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from helpers.config import PLOTS_ROOT

---
## Step 1: Feature Extraction

Extract stride-level and run-level kinematic measures from the preprocessed data. For each trial, strides around the belt-speed transition are identified and measures are computed per stride (e.g. stride duration, swing velocity, joint angles, body posture) and per run (e.g. behavioural metrics).

The full list of measures is defined in `helpers/config.py` (`measures_list`).

| | |
|---|---|
| **Scripts** | `apa_analysis/GetFeatures/BasicMeasures.py`, `MeasuresByStride.py`, `MeasuresByRun.py` |
| **Inputs** | Per-condition `.pkl` files from preprocessing |
| **Outputs** | Stride-level and run-level measure DataFrames (pickled) |

In [ ]:
from apa_analysis.GetFeatures.BasicMeasures import GetConditionFiles

# --- Extended experiments ---
for speed in ['LowHigh', 'LowMid', 'HighLow']:
    print(f"Extracting measures for Extended: {speed}...")
    GetConditionFiles(
        exp='APAChar', speed=speed, repeat_extend='Extended',
        analyses=["behaviour", "single", "multi"]
    ).get_dirs()

# --- Repeats (per-day) ---
for day in ['Day1', 'Day2', 'Day3']:
    print(f"Extracting measures for Repeats {day}...")
    GetConditionFiles(
        exp='APAChar', speed='LowHigh', repeat_extend='Repeats', exp_wash='Exp', day=day,
        analyses=["behaviour", "single", "multi"]
    ).get_dirs()

---
## Step 2: Main Characterisation Analysis

The main analysis pipeline performs:
1. **Data collection & normalisation** - Loads feature data, optionally normalises across conditions
2. **Feature clustering** - Groups correlated features using k-means clustering
3. **Single-feature predictions** - Tests individual features for phase-discriminating power via logistic regression
4. **PCA** - Dimensionality reduction on the full feature set
5. **Multi-feature predictions** - Logistic regression on PCs to predict experimental phase
6. **Cross-validation** - Evaluates prediction robustness

Analysis settings are configured in `apa_analysis/config.py`.

| | |
|---|---|
| **Script** | `apa_analysis/Characterisation/MainAnalysis/Main.py` |
| **Inputs** | Stride-level measure DataFrames from Step 1 |
| **Outputs** | PCA results, regression models, prediction accuracy, plots |

In [ ]:
from apa_analysis.Characterisation.MainAnalysis.Main import main as run_main_analysis
from apa_analysis.config import global_settings, condition_specific_settings, instance_settings

# Log flattened settings for reproducibility
global_settings["LowHigh_c"] = condition_specific_settings['APAChar_LowHigh']['c']
global_settings["HighLow_c"] = condition_specific_settings['APAChar_HighLow']['c']
global_settings["LowMid_c"] = condition_specific_settings['APAChar_LowMid']['c']
global_settings["LowHigh_mice"] = condition_specific_settings['APAChar_LowHigh']['global_fs_mouse_ids']
global_settings["HighLow_mice"] = condition_specific_settings['APAChar_HighLow']['global_fs_mouse_ids']
global_settings["LowMid_mice"] = condition_specific_settings['APAChar_LowMid']['global_fs_mouse_ids']

settings_to_log = {
    "global_settings": global_settings,
    "instance_settings": instance_settings
}

# Run each configured condition
for inst in instance_settings:
    run_main_analysis(
        global_settings["stride_numbers"],
        global_settings["phases"],
        condition=inst["condition"],
        exp=inst["exp"],
        day=inst["day"],
        compare_condition=inst["compare_condition"],
        settings_to_log=settings_to_log,
        residuals=global_settings["residuals"]
    )

---
## Step 3: Learning Analysis

Analyse how APAs develop over the course of the experiment. This includes:
- Identifying fast vs slow learners based on prediction accuracy trajectories
- Plotting PC values and predictions across trials/days
- Computing learning and extinction slopes
- Fitting regression models to individual PCs

Requires output from the Main analysis (Step 2) - specifically the LowHigh preprocessed data and PCA predictions.

| | |
|---|---|
| **Script** | `apa_analysis/Characterisation/Learning.py` |
| **Inputs** | PCA predictions and preprocessed feature data from Step 2 |
| **Outputs** | Learning curve plots, slope analysis, fast/slow learner classification |

In [ ]:
from apa_analysis.Characterisation.Learning import Learning

# --- Load LH prediction data from Main analysis output ---
LH_MultiFeatPath = PLOTS_ROOT + r"\LH_res_-3-2-1_APA2Wash2\APAChar_LowHigh_Extended\MultiFeaturePredictions"
LH_preprocessed_data_file_path = PLOTS_ROOT + r"\LH_res_-3-2-1_APA2Wash2\preprocessed_data_APAChar_LowHigh.pkl"
LH_stride_0_preprocessed_data_file_path = PLOTS_ROOT + r"\LH_LHpcsonly_res_0_APA2Wash2\preprocessed_data_APAChar_LowHigh.pkl"
LH_pred_path = f"{LH_MultiFeatPath}\\pca_predictions_APAChar_LowHigh.pkl"
LH_pca_path = f"{LH_MultiFeatPath}\\pca_APAChar_LowHigh.pkl"

with open(LH_preprocessed_data_file_path, 'rb') as f:
    LH_preprocessed_data = pickle.load(f)['feature_data']
with open(LH_stride_0_preprocessed_data_file_path, 'rb') as f:
    LH_stride0_preprocessed_data = pickle.load(f)['feature_data']
with open(LH_pred_path, 'rb') as f:
    LH_pred_data = pickle.load(f)
with open(LH_pca_path, 'rb') as f:
    LH_pca_data = pickle.load(f)

# --- Run learning analysis ---
save_dir = os.path.join(PLOTS_ROOT, 'Learning_1')
os.makedirs(save_dir, exist_ok=True)

chosen_pcs = [1, 3, 7]

learning = Learning(
    LH_preprocessed_data, LH_stride0_preprocessed_data,
    LH_pred_data, LH_pca_data, save_dir,
    disturb_pred_file=PLOTS_ROOT + r"\LH_subtract_res_0_APA1APA2-PCStot=60-PCSuse=12\APAChar_LowHigh_Extended\MultiFeaturePredictions\pca_predictions_APAChar_LowHigh.pkl"
)

learning.find_learned_trials(smoothing=None, phase='learning')
learning.find_learned_trials(smoothing=None, phase='extinction')
print("Fast learners:", learning.fast_learning)
print("Slow learners:", learning.slow_learning)

learning.plot_learning_by_extinction_scatter()
learning.plot_prediction_per_day_pcwise(chosen_pcs=chosen_pcs)
learning.plot_pcvalues_per_day_pcwise(chosen_pcs=chosen_pcs)
learning.plot_prediction_delta()
learning.plot_prediction_per_day(fs=7)
learning.compute_phase_slope_differences(chosen_pcs=chosen_pcs, smooth_window=3)
learning.compute_daybreak_slopes(chosen_pcs=chosen_pcs, smooth_window=3)
learning.plot_pc_correlations_heatmap(pcs=chosen_pcs, s=-1, fs=7)
learning.get_slopes_across_chunks(pcs=chosen_pcs, fs=7)
learning.plot_slopes_all_pcs(pcs=chosen_pcs)
learning.plot_total_predictions_x_trial(smooth_window=3)
learning.plot_line_important_pcs_x_trial(chosen_pcs=chosen_pcs, smooth_window=10)
learning.plot_line_important_pcs_preds_x_trial(chosen_pcs=chosen_pcs, smooth_window=10)
learning.fit_pcwise_regression_model(chosen_pcs=chosen_pcs)

for phase in ['APA1', 'APA2', 'APA']:
    learning.plot_disturbance_by_prediction_interpolated(phase)

---
## Step 4: When-APA Analysis

Analyse the stride-wise timing of APAs. Examines which strides before the transition carry APA-related signals by comparing model accuracy across stride positions and correlating PC weights and values across strides.

| | |
|---|---|
| **Script** | `apa_analysis/Characterisation/WhenAPA.py` |
| **Inputs** | PCA predictions and preprocessed feature data from Step 2 |
| **Outputs** | Stride-timing accuracy plots, PC correlation heatmaps, timeseries plots |

In [ ]:
from apa_analysis.Characterisation.WhenAPA import WhenAPA

save_dir = os.path.join(PLOTS_ROOT, 'WhenAPA_corrections')
os.makedirs(save_dir, exist_ok=True)

when_apa = WhenAPA(
    LH_preprocessed_data, LH_stride0_preprocessed_data,
    LH_pred_data, LH_pca_data, save_dir
)

when_apa.plot_accuracy_of_each_stride_model()
when_apa.plot_corr_pc_weights_heatmap()
when_apa.plot_corr_pcs_heatmap(pcs_to_plot=None)
when_apa.plot_corr_pcs_heatmap(pcs_to_plot=[0, 2, 6])
when_apa.plot_line_pcs_apa_vs_wash()
when_apa.plot_pcs_timeseries_by_stride()